# Notebook 1. Quantization basics

This notebook builds the primitives that every later notebook and every scheme in Pare rests on. You will implement uniform affine quantization, choose scales and zero-points, measure the error they cause, and see why the granularity of the scale matters as much as the bit-width. The last section runs the same operations through Pare and checks that the two implementations agree.

Prerequisites are basic PyTorch and some familiarity with neural network weights.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, "..")   # import Pare from this checkout; `pip install pare-quant` also works
torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. Why quantize

A 7-billion-parameter model in FP16 occupies $7 \times 10^9 \times 2$ bytes, about 14 GB of weights before any activations or KV cache. Storing the weights in fewer bits reduces that memory in proportion.

| Weight dtype | Bits | 7B model | 70B model |
|-------|------|----------|----------|
| FP32  | 32   | 28 GB    | 280 GB   |
| FP16 or BF16 | 16 | 14 GB | 140 GB  |
| INT8  | 8    | 7 GB     | 70 GB    |
| INT4  | 4    | 3.5 GB   | 35 GB    |

The memory saving comes from the storage format alone. A speed gain needs a matmul kernel that reads the packed weights directly; a model that dequantizes each weight matrix back to FP16 before every matmul, as Pare's reference forward pass does, is slower than FP16, not faster (see the throughput table in the README). Keep the two benefits apart when you read quantization claims.

## 2. Uniform affine quantization

A float value $x$ is mapped to an integer code $q$ with two parameters, a scale $s > 0$, which is the width of one step, and an integer zero-point $z$, which is the code that represents $0.0$.

$$q = \mathrm{clamp}\left(\left\lfloor \frac{x}{s} \right\rceil + z,\; q_{\min},\; q_{\max}\right), \qquad \hat{x} = s \,(q - z)$$

The rounding error $\hat{x} - x$ is at most $s/2$ in magnitude for values inside the range, and values outside the range are clipped.

Pare stores 2-, 3- and 4-bit codes on the unsigned grid $[0, 2^b - 1]$, so INT4 codes run from 0 to 15, and stores INT8 on the signed grid $[-128, 127]$. These notebooks use the same grids, so the code you write here produces the same integers as the library.

In [ ]:
def grid(bits: int) -> tuple[int, int]:
    # Pare's convention: unsigned codes below 8 bits, signed codes at 8 bits.
    if bits == 8:
        return -128, 127
    return 0, 2**bits - 1


def quantize(x: torch.Tensor, s: torch.Tensor, z: torch.Tensor, bits: int) -> torch.Tensor:
    q_min, q_max = grid(bits)
    return torch.clamp(torch.round(x / s + z), q_min, q_max)


def dequantize(q: torch.Tensor, s: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    return s * (q - z)


x = torch.tensor([-1.5, -0.8, 0.0, 0.3, 0.9, 1.2])
s, z = torch.tensor(0.2), torch.tensor(8.0)   # INT4: step 0.2, zero-point at the middle code
q = quantize(x, s, z, bits=4)
x_hat = dequantize(q, s, z)

print("x    ", x.tolist())
print("q    ", q.tolist())
print("x_hat", x_hat.tolist())
print("error", (x_hat - x).tolist())

With $s = 0.2$ and $z = 8$ the representable range is $[-1.6, 1.4]$, so $-1.5$ and $1.2$ are inside it and round to within $0.1$, while a value of $2.0$ would be clipped to the top code.

## 3. Choosing the scale and zero-point

Given a tensor, we want the range $[s(q_{\min} - z),\; s(q_{\max} - z)]$ to cover its values tightly. Two conventions are in use.

**Asymmetric.** Stretch the grid over $[x_{\min}, x_{\max}]$, after extending the range to include zero so that $0.0$ stays exactly representable.

$$s = \frac{x_{\max} - x_{\min}}{q_{\max} - q_{\min}}, \qquad z = \left\lfloor q_{\min} - \frac{x_{\min}}{s} \right\rceil$$

**Symmetric.** Centre the grid on zero with half-width $\max(|x_{\min}|, |x_{\max}|)$. On the signed INT8 grid this gives $z = 0$. On an unsigned grid the codes cannot be negative, so the zero-point sits at the middle code, $z = (q_{\max} + 1)/2$, which is 8 for INT4.

Asymmetric quantization uses every code when the data are skewed, as ReLU outputs are. Symmetric quantization saves the zero-point arithmetic in a kernel. Pare defaults to asymmetric (`sym=False`).

In [ ]:
def scale_asymmetric(x: torch.Tensor, bits: int, dim=None):
    q_min, q_max = grid(bits)
    x_min = x.amin(dim=dim, keepdim=dim is not None).clamp(max=0.0)   # range must include 0
    x_max = x.amax(dim=dim, keepdim=dim is not None).clamp(min=0.0)
    s = ((x_max - x_min) / (q_max - q_min)).clamp(min=1e-8)
    z = torch.round(q_min - x_min / s).clamp(q_min, q_max)
    return s, z


def scale_symmetric(x: torch.Tensor, bits: int, dim=None):
    q_min, q_max = grid(bits)
    abs_max = x.abs().amax(dim=dim, keepdim=dim is not None).clamp(min=1e-8)
    if q_min < 0:                                  # signed grid: zero-point 0
        s = abs_max / max(abs(q_min), q_max)
        z = torch.zeros_like(s)
    else:                                          # unsigned grid: zero-point at the middle code
        s = 2 * abs_max / (q_max - q_min)
        z = torch.full_like(s, (q_max + 1) / 2)
    return s, z


def round_trip(x, s, z, bits):
    return dequantize(quantize(x, s, z, bits), s, z)


w = torch.randn(1000) * 0.5
for bits in (8, 4):
    mae_sym = (round_trip(w, *scale_symmetric(w, bits), bits) - w).abs().mean()
    mae_asym = (round_trip(w, *scale_asymmetric(w, bits), bits) - w).abs().mean()
    print(f"INT{bits}  symmetric MAE {mae_sym:.5f}   asymmetric MAE {mae_asym:.5f}")

For a Gaussian vector the two conventions give nearly the same error, since its minimum and maximum have about the same magnitude. The asymmetric grid pays off on skewed data, which Exercise 2 asks you to check.

## 4. What the error looks like

Plotting $\hat{x}$ against $x$ shows the staircase. The step is $s$, the error is a sawtooth with amplitude $s/2$, and INT4 has sixteen steps where INT8 has 256.

In [ ]:
x_line = torch.linspace(-2, 2, 1000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, bits in zip(axes, (8, 4)):
    s, z = scale_asymmetric(x_line, bits)
    x_hat = round_trip(x_line, s, z, bits)
    ax.plot(x_line, x_line, "k--", lw=1, label="x")
    ax.plot(x_line, x_hat, "b-", lw=1.5, label="dequantized")
    ax.plot(x_line, x_hat - x_line, "r-", lw=1, alpha=0.7, label="error")
    ax.set_title(f"INT{bits}: {2**bits} levels, step {s.item():.4f}")
    ax.set_xlabel("x")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Granularity

One scale for a whole weight matrix lets a single large value set the step for every other value. Computing separate scales for subsets of the matrix confines the damage. For a weight matrix $W \in \mathbb{R}^{O \times I}$ three choices are common.

| Granularity | Number of scales | Scope of each scale |
|-------------|------------------|---------------------|
| Per tensor  | 1                | All $O \times I$ values |
| Per channel | $O$              | One output row |
| Per group   | $O \times I / g$ | $g$ consecutive input columns within a row |

GPTQ and AWQ report their INT4 results with groups of 128, and Pare's benchmarks use the same value. Smaller groups give lower error and more scale storage.

In [ ]:
def quantize_per_tensor(W, bits):
    s, z = scale_asymmetric(W, bits)
    return round_trip(W, s, z, bits)


def quantize_per_channel(W, bits):
    s, z = scale_asymmetric(W, bits, dim=1)    # one scale per row, shape [n_out, 1]
    return round_trip(W, s, z, bits)


def quantize_per_group(W, bits, group_size=128):
    n_out, n_in = W.shape
    Wg = W.reshape(n_out, n_in // group_size, group_size)   # [n_out, n_groups, g]
    s, z = scale_asymmetric(Wg, bits, dim=2)   # one scale per group, shape [n_out, n_groups, 1]
    return round_trip(Wg, s, z, bits).reshape(n_out, n_in)


torch.manual_seed(0)
W = torch.randn(32, 256) * 0.3
W[5, 10] = 3.5     # one outlier in row 5
W[12, 80] = -4.0   # one outlier in row 12

for bits in (8, 4):
    for name, fn in [("per tensor", quantize_per_tensor),
                     ("per channel", quantize_per_channel),
                     ("per group (128)", quantize_per_group)]:
        mae = (fn(W, bits) - W).abs().mean()
        print(f"INT{bits} {name:>16}: MAE = {mae:.5f}")
    print()

At INT8 the three granularities are close, because 256 levels absorb an outlier. At INT4 the per-tensor error is several times the per-group error: the two outliers set the step for all 8192 weights, while in the per-group case they affect only their own groups of 128.

## 6. Quantization noise and the 6 dB per bit rule

For values well inside the range, the rounding error is close to uniform on $[-s/2, s/2]$, whose variance is $s^2/12$. Each extra bit halves $s$ and so divides the mean squared error by four, which is a 6 dB gain in signal-to-noise ratio.

In [ ]:
w = torch.randn(100_000)
bits_range = range(2, 9)
mse_values = []
for bits in bits_range:
    s, z = scale_symmetric(w, bits)
    mse = (round_trip(w, s, z, bits) - w).pow(2).mean().item()
    mse_values.append(mse)
    snr_db = 10 * np.log10(w.pow(2).mean().item() / mse)
    print(f"INT{bits}: MSE = {mse:.2e}   SNR = {snr_db:5.1f} dB   s^2/12 = {s.item()**2 / 12:.2e}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(list(bits_range), mse_values, "o-")
ax.set_xlabel("bits")
ax.set_ylabel("MSE")
ax.set_title("MSE against bit-width")
ax.grid(True, which="both", alpha=0.4)
plt.tight_layout()
plt.show()

The measured MSE matches $s^2/12$ from 3 bits upwards. At 2 bits it is smaller than the formula predicts, because nearly every value falls into the two middle cells and the rounding error is no longer uniform across a cell.

## 7. A realistic weight tensor

Trained LLM weights are close to Gaussian with a standard deviation of a few hundredths (Notebook 6 plots a real GPT-2 layer). The histograms below quantize one such vector at three settings and overlay the dequantized values on the originals.

In [ ]:
torch.manual_seed(7)
w_real = torch.randn(4096) * 0.05

settings = [
    ("INT8, one scale", lambda w: round_trip(w, *scale_asymmetric(w, 8), 8)),
    ("INT4, one scale", lambda w: round_trip(w, *scale_asymmetric(w, 4), 4)),
    ("INT4, groups of 128", lambda w: quantize_per_group(w.reshape(1, -1), 4).reshape(-1)),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, fn) in zip(axes, settings):
    w_hat = fn(w_real)
    mae = (w_hat - w_real).abs().mean()
    ax.hist(w_real.numpy(), bins=80, alpha=0.5, label="FP32")
    ax.hist(w_hat.numpy(), bins=80, alpha=0.5, label="dequantized")
    ax.set_title(f"{title}\nMAE = {mae:.5f}")
    ax.set_xlabel("weight value")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. The same operations in Pare

`pare.core` exposes these primitives as `compute_scale`, `quantize_tensor` and `dequantize_tensor`. The cell below runs the notebook's per-group asymmetric quantizer and Pare's on the same matrix and checks that they produce identical codes and identical reconstructions. If the assertion holds, everything above is exactly what the library does inside `QuantizedLinear`.

In [ ]:
from pare.core.dtype import QuantDtype
from pare.core.functional import dequantize_tensor, quantize_tensor
from pare.core.scale import compute_scale

torch.manual_seed(1)
W = torch.randn(64, 256) * 0.05

# Pare
scale, zero = compute_scale(W, QuantDtype.INT4, granularity="per_group", group_size=128)
Q_pare = quantize_tensor(W, scale, zero, QuantDtype.INT4)   # [64, 2, 128] int32 codes
# quantize_tensor returns grouped codes; dequantize_tensor expects them flattened to [out, in].
W_pare = dequantize_tensor(Q_pare.reshape(64, -1), scale, zero).reshape(W.shape)

# This notebook
Wg = W.reshape(64, 2, 128)
s, z = scale_asymmetric(Wg, 4, dim=2)
Q_nb = quantize(Wg, s, z, 4)
W_nb = dequantize(Q_nb, s, z).reshape(W.shape)

assert torch.equal(Q_pare.float(), Q_nb), "codes differ"
assert torch.allclose(W_pare, W_nb), "reconstructions differ"
print(f"scale shape {tuple(scale.shape)}, zero-point range [{zero.min():.0f}, {zero.max():.0f}]")
print(f"MAE {(W_pare - W).abs().mean():.6f}; the notebook and Pare agree exactly.")

## Exercises

1. Set `group_size` to 32 and 512 in Section 5 and record the INT4 MAE for each. Then compute the storage overhead of the scales for each setting, assuming FP16 scales, as a fraction of the INT4 weight bytes.
2. Replace the Gaussian vector in Section 6 with `torch.rand(100_000)`, which is uniform on $[0, 1)$. Which of the two conventions in Section 3 gives the lower INT4 error, and why?
3. In Section 8, change `granularity` to `"per_channel"` and make the notebook implementation agree again.

## Summary

| Concept | Formula | Note |
|---------|---------|------|
| Quantize | $q = \mathrm{clamp}(\lfloor x/s \rceil + z,\, q_{\min},\, q_{\max})$ | Codes are unsigned below 8 bits in Pare |
| Dequantize | $\hat{x} = s (q - z)$ | Error at most $s/2$ inside the range |
| Asymmetric scale | $s = (x_{\max} - x_{\min}) / (q_{\max} - q_{\min})$ | Pare's default |
| Symmetric scale | half-width $\max\lvert x \rvert$, $z$ at 0 or the middle code | Saves zero-point arithmetic |
| Noise | $\mathrm{MSE} \approx s^2/12$ | 6 dB per bit |
| Granularity | per tensor, per channel, per group | Groups of 128 for INT4 in GPTQ, AWQ and Pare |

Notebook 2 turns these primitives into round-to-nearest quantization of a whole model, with packed INT4 storage.